# GridLock — Congestion-Impact Score & Models

**Prerequisite:** run `gridlock_eda.ipynb` first (produces `cleaned.parquet` and `derived/severity.parquet` in `/kaggle/working`).

Pipeline: prepare → features (+EB smoothing) → spatial-lag/KDE → Getis-Ord Gi* → PCA → ensemble impact score (β=0.75) → validation → outputs → supervised impact model → hotspot-detection classifier. See `fe/findings/FE_REPORT.md` and `fe/findings/MODEL_CARD.md`.

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","pygeohash"],check=True)
os.makedirs("/kaggle/working/derived",exist_ok=True)
os.makedirs("/kaggle/working/fe_out",exist_ok=True)
# write scorelib.py so cells can import it
open("/kaggle/working/scorelib.py","w").write('# fe/scorelib.py\nimport numpy as np, pandas as pd\ndef pct(s):  # percentile rank 0..1\n    return pd.Series(s).rank(pct=True).values\n\ndef impact_character(f):\n    """Volume-INDEPENDENT impact character: rates of carriageway-blocking / heavy / road-context.\n    Prefers empirical-Bayes-smoothed rates (*_eb) when present (small-cell noise reduction)."""\n    t3 = f["tier3_share_eb"] if "tier3_share_eb" in f else f["tier3_share"]\n    hv = f["heavy_share_eb"] if "heavy_share_eb" in f else f["heavy_share"]\n    road=(f["f_main_road"].values + f["f_junction"].values + f["f_circle"].values).clip(0,1) \\\n         if "f_circle" in f else (f["f_main_road"].values + f["f_junction"].values)\n    return (pct(t3) + pct(hv) + pct(road)) / 3.0\n\ndef ensemble_impact(gi_z, char, beta=0.5):\n    """Blend Gi* impact-significance (volume-aware) with impact-character (volume-independent)."""\n    raw = beta*pct(gi_z) + (1.0-beta)*np.asarray(char)\n    return 100.0*pct(raw)\n\n# --- cross-checks (not the shipped score) ---\ndef hand_composite(f):\n    comp={"Volume":pct(f["n"]),"Severity":pct(f["sev_sum_total"]),\n          "VehicleImpact":pct(f["heavy_share"]),"RoadContext":pct(f["f_main_road"]+f["f_junction"]),\n          "Persistence":pct(f["recurrence_weeks"])}\n    w={"Volume":.25,"Severity":.30,"VehicleImpact":.20,"RoadContext":.15,"Persistence":.10}\n    return 100.0*pct(sum(w[k]*comp[k] for k in w))\n')
print("setup done")

## `00_prepare.py`

In [ ]:
import pandas as pd, numpy as np, pygeohash as pgh
df=pd.read_parquet("/kaggle/working/cleaned.parquet")
sev=pd.read_parquet("/kaggle/working/derived/severity.parquet")
n0=len(df)
df=df.merge(sev[["id","max_sev","sev_sum"]],on="id",how="left")
# GUARD 1: severity join conserves rows and is complete
assert len(df)==n0 and df["max_sev"].notna().all(), "severity join lost rows or has NaN"
# is_valid already on cleaned.parquet; filter
base=df[df["is_valid"]].copy()
print("is_valid retained: %d / %d (%.1f%%)"%(len(base),n0,len(base)/n0*100))
# coalesce vehicle type
ut=base["updated_vehicle_type"]
base["vehicle_type_final"]=ut.where(~(ut.isin(["NULL",""])|ut.isna()), base["vehicle_type"])
# GUARD 2: no null vehicle_type_final
assert base["vehicle_type_final"].notna().all(), "vehicle_type_final has nulls"
# geohash encode (gh7 primary, gh6/gh5 rollups)
base["gh7"]=[pgh.encode(la,lo,precision=7) for la,lo in zip(base["latitude"],base["longitude"])]
base["gh6"]=base["gh7"].str[:6]; base["gh5"]=base["gh7"].str[:5]
# GUARD 3: geohash length correct
assert (base["gh7"].str.len()==7).all(), "gh7 wrong length"
base.to_parquet("/kaggle/working/fe_base.parquet")
print("fe_base rows:",len(base),"| distinct gh7:",base["gh7"].nunique(),
      "| gh6:",base["gh6"].nunique(),"| gh5:",base["gh5"].nunique())


## `10_features.py`

In [ ]:
import pandas as pd, numpy as np
b=pd.read_parquet("/kaggle/working/fe_base.parquet")
b["created_dt"]=pd.to_datetime(b["created_dt"],utc=True)
b["isoweek"]=b["created_dt"].dt.isocalendar().week  # nullable UInt32; nunique ignores NaT
b["date"]=b["created_dt"].dt.date
HEAVY={"BUS (BMTC/KSRTC)","PRIVATE BUS","TEMPO","HGV","LORRY/GOODS VEHICLE","TANKER","FACTORY BUS","TOURIST BUS","SCHOOL VEHICLE"}
COMMERCIAL=HEAVY|{"PASSENGER AUTO","GOODS AUTO","LGV","MAXI-CAB","VAN"}
TWOW={"SCOOTER","MOTOR CYCLE","MOPED"}
# per-ticket impact intensity: steep tier weights x vehicle multiplier (impact, not volume)
TIER_W={0:0.0,1:0.5,2:1.0,3:6.0}
b["_tw"]=b["max_sev"].map(TIER_W)
b["_vmult"]=np.where(b["vehicle_type_final"].isin(HEAVY),2.0,
              np.where(b["vehicle_type_final"].isin(COMMERCIAL),1.3,1.0))
b["impact_intensity"]=b["_tw"]*b["_vmult"]
loc=b["location"].fillna("").str.lower()
b["f_main_road"]=loc.str.contains("main road",regex=False)
b["f_circle"]=loc.str.contains("circle",regex=False)
b["f_cross"]=loc.str.contains("cross",regex=False)
b["f_junction"]=loc.str.contains("junction",regex=False)
b["f_busstop_school_hosp"]=loc.str.contains("bus stop|busstop|school|hospital",regex=True)
b["f_metro"]=loc.str.contains("metro",regex=False)
b["f_market"]=loc.str.contains("market",regex=False)
b["f_mall"]=loc.str.contains("mall",regex=False)
g=b.groupby("gh7")
feat=pd.DataFrame({
 "n":g.size(),
 "n_valid":g.size(),
 "distinct_days":g["date"].nunique(),
 "distinct_weeks":g["isoweek"].nunique(),
 "lat":g["latitude"].mean(),"lon":g["longitude"].mean(),
 "gh6":g["gh6"].first(),"gh5":g["gh5"].first(),
 "sev_sum_total":g["sev_sum"].sum(),
 "impact_intensity_total":g["impact_intensity"].sum(),
 "mean_sev":g["sev_sum"].mean(),
 "tier3_count":g["max_sev"].apply(lambda s:(s>=3).sum()),
 "tier3_share":g["max_sev"].apply(lambda s:(s>=3).mean()),
 "heavy_share":g["vehicle_type_final"].apply(lambda s:s.isin(HEAVY).mean()),
 "commercial_share":g["vehicle_type_final"].apply(lambda s:s.isin(COMMERCIAL).mean()),
 "twowheeler_share":g["vehicle_type_final"].apply(lambda s:s.isin(TWOW).mean()),
 "f_main_road":g["f_main_road"].mean(),"f_circle":g["f_circle"].mean(),
 "f_cross":g["f_cross"].mean(),"f_junction":g["f_junction"].mean(),
 "f_busstop_school_hosp":g["f_busstop_school_hosp"].mean(),
 "f_metro":g["f_metro"].mean(),"f_market":g["f_market"].mean(),"f_mall":g["f_mall"].mean(),
 "distinct_devices":g["device_id"].nunique(),
 "distinct_officers":g["created_by_id"].nunique(),
 "recurrence_weeks":g["isoweek"].nunique(),
}).reset_index()
# GUARD: cell counts sum to base rows; tier3_share in [0,1]
assert feat["n"].sum()==len(b), "feature counts != base rows"
assert feat["tier3_share"].between(0,1).all(), "tier3_share out of range"
# Empirical-Bayes shrinkage of small-cell rates toward the global mean (K pseudo-tickets):
# reduces noise from tiny cells whose tier3/heavy rates are unreliable (improves stability).
K_EB=40
m_t3=feat["tier3_count"].sum()/feat["n"].sum()
heavy_count=feat["heavy_share"]*feat["n"]
m_h=heavy_count.sum()/feat["n"].sum()
feat["tier3_share_eb"]=(feat["tier3_count"]+K_EB*m_t3)/(feat["n"]+K_EB)
feat["heavy_share_eb"]=(heavy_count+K_EB*m_h)/(feat["n"]+K_EB)
feat["ranked"]=feat["n_valid"]>=50   # headline ranking needs real evidence
feat.to_parquet("/kaggle/working/cell_features.parquet")
print("cells:",len(feat),"| ranked(>=25):",int(feat["ranked"].sum()))
print(feat[["n","tier3_share","heavy_share","f_main_road"]].describe().round(3).to_string())


## `11_spatial_lag_kde.py`

In [ ]:
import pandas as pd, numpy as np
from libpysal.weights import KNN
from scipy.stats import gaussian_kde
f=pd.read_parquet("/kaggle/working/cell_features.parquet")
n_in=len(f)
xy=f[["lon","lat"]].values
w=KNN.from_array(xy,k=8); w.transform="r"
def lag(col):
    a=f[col].values
    return np.array([sum(wt*a[nb] for wt,nb in zip(w.weights[i],w.neighbors[i])) for i in range(len(f))])
f["lag_n"]=lag("n"); f["lag_sev_sum"]=lag("sev_sum_total"); f["lag_tier3_share"]=lag("tier3_share")
# severity-weighted KDE evaluated at centroids
pts=np.vstack([f["lon"],f["lat"]])
kde=gaussian_kde(pts,weights=f["sev_sum_total"].values,bw_method=0.05)
f["kde_sev"]=kde(pts)
# interactions
f["heavy_x_mainroad"]=f["heavy_share"]*f["f_main_road"]
f["tier3_x_junction"]=f["tier3_share"]*f["f_junction"]
# GUARD: no NaN introduced; lengths preserved
assert len(f)==n_in, "row count changed"
assert f[["lag_n","lag_sev_sum","kde_sev","heavy_x_mainroad"]].notna().all().all(), "NaN in new features"
f.to_parquet("/kaggle/working/cell_features_full.parquet")
print("added spatial-lag/KDE/interactions; cells:",len(f),"cols:",f.shape[1])
print(f[["lag_sev_sum","kde_sev","heavy_x_mainroad","tier3_x_junction"]].describe().round(4).to_string())


## `20_gistar.py`

In [ ]:
import pandas as pd, numpy as np
from libpysal.weights import KNN
from esda.getisord import G_Local
f=pd.read_parquet("/kaggle/working/cell_features_full.parquet")
xy=f[["lon","lat"]].values
w=KNN.from_array(xy,k=8); w.transform="r"
y=f["impact_intensity_total"].values.astype(float)   # Gi* on IMPACT intensity, not volume
gi=G_Local(y,w,star=True,seed=42)   # Gi* (include self)
f["gi_z"]=gi.Zs
f["gi_p"]=gi.p_sim
# GUARD: output length matches cells; z-scores finite
assert len(gi.Zs)==len(f), "Gi* length mismatch"
assert np.isfinite(f["gi_z"]).all(), "Gi* produced non-finite z"
f.to_parquet("/kaggle/working/cell_features_full.parquet")
sig=(f["gi_p"]<0.05)&(f["gi_z"]>0)
print("significant hot cells (p<0.05, z>0): %d / %d"%(int(sig.sum()),len(f)))
print(f.sort_values("gi_z",ascending=False)[["gh7","n","tier3_share","gi_z","gi_p"]].head(10).round(3).to_string(index=False))


## `21_pca.py`

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
f=pd.read_parquet("/kaggle/working/cell_features_full.parquet")
PCA_FEATS=["n","sev_sum_total","tier3_share","heavy_share","commercial_share",
           "f_main_road","f_circle","f_junction","f_busstop_school_hosp",
           "lag_sev_sum","lag_tier3_share","kde_sev","heavy_x_mainroad","tier3_x_junction",
           "distinct_devices","recurrence_weeks"]
X=StandardScaler().fit_transform(f[PCA_FEATS].fillna(0).values)
p=PCA(n_components=5,random_state=42).fit(X)
scores=p.transform(X)
pc1=scores[:,0]
if np.corrcoef(pc1,f["sev_sum_total"])[0,1]<0: pc1=-pc1
evr=p.explained_variance_ratio_
if evr[0]<0.40:
    pc2=scores[:,1]
    if np.corrcoef(pc2,f["sev_sum_total"])[0,1]<0: pc2=-pc2
    composite=(evr[0]*pc1+evr[1]*pc2)/(evr[0]+evr[1])
    print("PC1 weak (<0.40) -> blended PC1+PC2")
else:
    composite=pc1
f["pca_composite"]=composite
# GUARD: composite finite, length matches
assert np.isfinite(f["pca_composite"]).all() and len(f)==len(f), "pca composite bad"
f.to_parquet("/kaggle/working/cell_features_full.parquet")
load=pd.Series(p.components_[0],index=PCA_FEATS).sort_values(key=abs,ascending=False)
print("PC1 explained var: %.3f | PC1-2: %.3f"%(evr[0],evr[:2].sum()))
print("PC1 loadings (|desc|):")
print(load.round(3).to_string())


## `22_ensemble.py`

In [ ]:
import pandas as pd, numpy as np, sys
sys.path.insert(0,"/kaggle/working")
import scorelib, importlib; importlib.reload(scorelib)
from scorelib import ensemble_impact, impact_character, hand_composite, pct
f=pd.read_parquet("/kaggle/working/cell_features_full.parquet")
BETA=0.75  # blend weight Gi*-significance vs impact-character; selected in 30_validate
           # (max ranked-cell stability s.t. tier3>=2.5 & heavy>=1.2; EB-smoothed rates, K=40)
f["impact_char"]=impact_character(f)
f["impact"]=ensemble_impact(f["gi_z"].values,f["impact_char"].values,beta=BETA)
f["hand_score"]=hand_composite(f)
f["rank"]=f["impact"].rank(ascending=False).astype(int)
# GUARD: impact in [0,100]
assert f["impact"].between(0,100).all(), "impact out of range"
f.to_parquet("/kaggle/working/cell_scores.parquet")
top=f[f["ranked"]].sort_values("impact",ascending=False).head(15)
print(top[["gh7","n","tier3_share","heavy_share","gi_z","impact_char","impact"]].round(3).to_string(index=False))


## `30_validate.py`

In [ ]:
import pandas as pd, numpy as np, sys, warnings; warnings.filterwarnings("ignore")
from scipy.stats import spearmanr, kendalltau
sys.path.insert(0,"/kaggle/working")
import scorelib, importlib; importlib.reload(scorelib)
from scorelib import ensemble_impact, impact_character, hand_composite, pct
from libpysal.weights import KNN
from esda.getisord import G_Local
b=pd.read_parquet("/kaggle/working/fe_base.parquet")
full=pd.read_parquet("/kaggle/working/cell_scores.parquet")
HEAVY=["BUS (BMTC/KSRTC)","PRIVATE BUS","TEMPO","HGV","LORRY/GOODS VEHICLE","TANKER","FACTORY BUS","TOURIST BUS","SCHOOL VEHICLE"]
COMM=HEAVY+["PASSENGER AUTO","GOODS AUTO","LGV","MAXI-CAB","VAN"]
TIER_W={0:0.0,1:0.5,2:1.0,3:6.0}

def build_scores(sub,beta=0.5,min_n=10):
    sub=sub.copy()
    sub["_ii"]=sub["max_sev"].map(TIER_W)*np.where(sub["vehicle_type_final"].isin(HEAVY),2.0,
                  np.where(sub["vehicle_type_final"].isin(COMM),1.3,1.0))
    loc=sub["location"].fillna("").str.lower()
    sub["_mr"]=loc.str.contains("main road",regex=False); sub["_jn"]=loc.str.contains("junction",regex=False)
    sub["_ci"]=loc.str.contains("circle",regex=False)
    g=sub.groupby("gh7")
    d=pd.DataFrame({"impact_intensity_total":g["_ii"].sum(),"lat":g["latitude"].mean(),"lon":g["longitude"].mean(),
        "n":g.size(),"tier3_share":g["max_sev"].apply(lambda s:(s>=3).mean()),
        "heavy_share":g["vehicle_type_final"].apply(lambda s:s.isin(HEAVY).mean()),
        "f_main_road":g["_mr"].mean(),"f_junction":g["_jn"].mean(),"f_circle":g["_ci"].mean()}).reset_index()
    d=d[d["n"]>=min_n].reset_index(drop=True)
    # EB smoothing (match 10_features): shrink rates toward global mean, K=25
    K=25; t3c=d["tier3_share"]*d["n"]; hvc=d["heavy_share"]*d["n"]
    mt3=t3c.sum()/d["n"].sum(); mh=hvc.sum()/d["n"].sum()
    d["tier3_share_eb"]=(t3c+K*mt3)/(d["n"]+K); d["heavy_share_eb"]=(hvc+K*mh)/(d["n"]+K)
    xy=d[["lon","lat"]].values; w=KNN.from_array(xy,k=min(8,len(d)-1)); w.transform="r"
    giz=G_Local(d["impact_intensity_total"].values.astype(float),w,star=True,seed=42).Zs
    d["impact"]=ensemble_impact(giz,impact_character(d),beta=beta)
    return d.set_index("gh7")["impact"]

b["created_dt"]=pd.to_datetime(b["created_dt"],utc=True)
half1=b[b["created_dt"]<"2024-02-01"]; half2=b[b["created_dt"]>="2024-02-01"]

# beta tuning: stability (Spearman half1 vs half2 at that beta) + face validity per beta
ranked_gh=set(full[full["ranked"]]["gh7"])   # well-supported cells (n>=50) — the ones we act on
print("beta | stab_all | stab_ranked | tier3_lift | heavy_lift")
base_t3=full["tier3_share"].mean(); base_h=full["heavy_share"].mean()
best=None
for beta in [0.25,0.4,0.5,0.6,0.75]:
    s1=build_scores(half1,beta); s2=build_scores(half2,beta)
    common=s1.index.intersection(s2.index); rho=spearmanr(s1[common],s2[common]).correlation
    rc=[g for g in common if g in ranked_gh]; rho_r=spearmanr(s1[rc],s2[rc]).correlation
    imp=ensemble_impact(full["gi_z"].values,full["impact_char"].values,beta=beta)
    t50=full.assign(_i=imp)[full["ranked"]].sort_values("_i",ascending=False).head(50)
    t3l=t50["tier3_share"].mean()/base_t3; hl=t50["heavy_share"].mean()/max(base_h,1e-9)
    print(f"{beta:.2f} |  {rho:.3f}  |   {rho_r:.3f}    |   {t3l:.2f}x   |  {hl:.2f}x")
    # select: max RANKED-cell stability among betas that deliver impact (tier3>=2.5, heavy>=1.2)
    if t3l>=2.5 and hl>=1.2 and (best is None or rho_r>best[1]): best=(beta,rho_r,t3l,hl)
print("SELECTED beta=%.2f (max ranked-stability s.t. tier3>=2.5 & heavy>=1.2): stab_ranked=%.3f, tier3=%.2fx, heavy=%.2fx"%best)

# concordance / bias on shipped score
top=full[full["ranked"]]; W=lambda a,c: kendalltau(a,c).correlation
print("concordance tau: Gi*-char=%.3f char-hand=%.3f Gi*-hand=%.3f"%(
    W(top["gi_z"],top["impact_char"]),W(top["impact_char"],top["hand_score"]),W(top["gi_z"],top["hand_score"])))
top50=full[full["ranked"]].sort_values("impact",ascending=False).head(50)
print("top-50 distinct_devices median=%.0f min=%.0f"%(top50["distinct_devices"].median(),top50["distinct_devices"].min()))
appr=b[b["validation_status_clean"]=="approved"]
sa=build_scores(appr,best[0]); ca=sa.index.intersection(full["gh7"])
rho_appr=spearmanr(sa[ca],pd.Series(full.set_index("gh7")["impact"]).reindex(ca)).correlation
print("approved-only sensitivity Spearman = %.3f"%rho_appr)
print("FACE VALIDITY (shipped): tier3 %.3f->%.3f (%.2fx) | heavy %.3f->%.3f (%.2fx)"%(
    base_t3,top50["tier3_share"].mean(),top50["tier3_share"].mean()/base_t3,
    base_h,top50["heavy_share"].mean(),top50["heavy_share"].mean()/max(base_h,1e-9)))
report=f"""# FE Validation Report (impact-intensity engine, EB-smoothed)
- Selected beta = {best[0]:.2f}
- Temporal stability (ranked cells, n>=50) Spearman = {best[1]:.3f}
- Approved-only sensitivity Spearman = {rho_appr:.3f} (threshold >=0.75)
- Top-50 median distinct_devices = {top50['distinct_devices'].median():.0f}
- Face validity: tier3 lift {top50['tier3_share'].mean()/base_t3:.2f}x | heavy lift {top50['heavy_share'].mean()/max(base_h,1e-9):.2f}x
- Concordance tau Gi*-char = {W(top['gi_z'],top['impact_char']):.3f}
"""
open("/kaggle/working/fe_out/validation_report.md","w").write(report)
print("\n"+report)


## `40_outputs.py`

In [ ]:
import pandas as pd, numpy as np, json
import pygeohash as pgh
f=pd.read_parquet("/kaggle/working/cell_scores.parquet")
def cell_polygon(gh):
    la,lo,dla,dlo=pgh.decode_exactly(gh)
    return [[lo-dlo,la-dla],[lo+dlo,la-dla],[lo+dlo,la+dla],[lo-dlo,la+dla],[lo-dlo,la-dla]]
feats=[]
for _,r in f.iterrows():
    feats.append({"type":"Feature","geometry":{"type":"Polygon","coordinates":[cell_polygon(r["gh7"])]},
        "properties":{"gh7":r["gh7"],"impact":round(float(r["impact"]),2),"rank":int(r["rank"]),
            "n":int(r["n"]),"tier3_share":round(float(r["tier3_share"]),3),
            "heavy_share":round(float(r["heavy_share"]),3),"gi_z":round(float(r["gi_z"]),2),
            "ranked":bool(r["ranked"])}})
gj={"type":"FeatureCollection","features":feats}
json.dump(gj,open("/kaggle/working/fe_out/cells.geojson","w"))
# GUARD: feature count == cell count
assert len(gj["features"])==len(f), "geojson feature count mismatch"
for k in ["gh6","gh5"]:
    roll=f.groupby(k).agg(impact_mean=("impact","mean"),impact_max=("impact","max"),
                          n=("n","sum"),lat=("lat","mean"),lon=("lon","mean")).reset_index()
    roll.to_csv(f"/kaggle/working/fe_out/rollup_{k}.csv",index=False)
prio=f[f["ranked"]].sort_values("impact",ascending=False).head(100)[
    ["rank","gh7","lat","lon","impact","n","tier3_share","heavy_share","f_main_road","f_junction","gi_z","distinct_devices"]]
prio.to_csv("/kaggle/working/fe_out/priority_table.csv",index=False)
f[["lon","lat","kde_sev","impact"]].to_csv("/kaggle/working/fe_out/kde_points.csv",index=False)
print("wrote cells.geojson (%d features), rollup_gh6/gh5.csv, priority_table.csv, kde_points.csv"%len(feats))
print("\nTOP-10 PRIORITY ZONES:")
print(prio.head(10)[["rank","gh7","lat","lon","impact","n","tier3_share","heavy_share"]].round(3).to_string(index=False))


## `41_map.py`

In [ ]:
import pandas as pd, folium, json
gj=json.load(open("/kaggle/working/fe_out/cells.geojson"))
f=pd.read_parquet("/kaggle/working/cell_scores.parquet")
# only render ranked cells (confidence) to keep the map meaningful
ranked_gh=set(f[f["ranked"]]["gh7"])
gj_r={"type":"FeatureCollection","features":[ft for ft in gj["features"] if ft["properties"]["gh7"] in ranked_gh]}
m=folium.Map(location=[12.97,77.59],zoom_start=12,tiles="cartodbpositron")
folium.Choropleth(geo_data=gj_r,data=f[f["ranked"]],columns=["gh7","impact"],
    key_on="feature.properties.gh7",fill_color="YlOrRd",fill_opacity=0.75,line_opacity=0.15,
    legend_name="Congestion-Impact Score (0-100)").add_to(m)
# markers for top-15 with decomposition popups
for _,r in f[f["ranked"]].sort_values("impact",ascending=False).head(15).iterrows():
    folium.CircleMarker([r["lat"],r["lon"]],radius=6,color="black",fill=True,fill_color="red",fill_opacity=0.9,
        popup=f"#{int(r['rank'])} {r['gh7']}: impact={r['impact']:.1f}, n={int(r['n'])}, T3={r['tier3_share']:.0%}, heavy={r['heavy_share']:.0%}").add_to(m)
m.save("/kaggle/working/fe_out/impact_map.html")
print("saved impact_map.html with %d ranked cells"%len(gj_r["features"]))


## `50_model_impact.py`

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import lightgbm as lgb
f=pd.read_parquet("/kaggle/working/cell_scores.parquet")  # features + impact/gi_z/impact_char
# train/eval on cells with enough evidence for a reliable target
d=f[f["n"]>=20].reset_index(drop=True)
groups=d["gh5"].astype("category").cat.codes.values
y=d["impact"].values

SCORE_INTERNAL={"impact","gi_z","gi_p","pca_composite","impact_char","hand_score","rank","ranked",
                "gh7","gh6","gh5","lat","lon"}
SPATIAL={"lag_n","lag_sev_sum","lag_tier3_share","kde_sev"}
allfeat=[c for c in d.columns if c not in SCORE_INTERNAL and d[c].dtype!=object]
FULL=allfeat
TRANSFER=[c for c in allfeat if c not in SPATIAL]   # cell-intrinsic only (no spatial leakage)
print("FULL feats:",len(FULL),"| TRANSFER feats:",len(TRANSFER),"| eval cells:",len(d),"| gh5 groups:",len(set(groups)))

def cv(feats,label):
    oof=np.zeros(len(d)); gkf=GroupKFold(n_splits=5)
    for tr,te in gkf.split(d,y,groups):
        m=lgb.LGBMRegressor(n_estimators=400,learning_rate=0.05,num_leaves=31,
            subsample=0.8,colsample_bytree=0.8,min_child_samples=20,reg_lambda=1.0,
            random_state=42,verbose=-1)
        m.fit(d.iloc[tr][feats],y[tr])
        oof[te]=m.predict(d.iloc[te][feats])
    r2=r2_score(y,oof); rho=spearmanr(y,oof).correlation
    print(f"  [{label:9s}] spatial-CV R2={r2:.3f}  Spearman={rho:.3f}")
    return oof,r2,rho

# baseline: volume only
oof_v,r2_v,_=cv(["n"],"volume")
oof_t,r2_t,rho_t=cv(TRANSFER,"transfer")
oof_f,r2_f,rho_f=cv(FULL,"full")

# feature importance from a full-data model (gain)
m=lgb.LGBMRegressor(n_estimators=400,learning_rate=0.05,num_leaves=31,subsample=0.8,
    colsample_bytree=0.8,min_child_samples=20,reg_lambda=1.0,random_state=42,verbose=-1)
m.fit(d[FULL],y)
imp=pd.Series(m.booster_.feature_importance(importance_type="gain"),index=FULL).sort_values(ascending=False)
imp.to_csv("/kaggle/working/fe_out/model_impact_importance.csv")
d.assign(pred_full=oof_f)[["gh7","impact","pred_full"]].to_csv("/kaggle/working/fe_out/model_impact_oof.csv",index=False)
m.booster_.save_model("/kaggle/working/fe_out/model_impact.txt")
print("\nTOP 12 IMPACT DRIVERS (gain):")
print(imp.head(12).round(0).to_string())
print("\nSUMMARY: volume-only R2=%.3f | transferable R2=%.3f | full R2=%.3f"%(r2_v,r2_t,r2_f))


## `51_model_detect.py`

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
f=pd.read_parquet("/kaggle/working/cell_scores.parquet")
d=f[f["n"]>=20].reset_index(drop=True)
groups=d["gh5"].astype("category").cat.codes.values
# label: high-IMPACT hotspot = top quartile of impact among busy cells
thr=d["impact"].quantile(0.75)
y=(d["impact"]>=thr).astype(int).values
print("hotspot label: impact>=%.1f | positives=%d/%d (%.1f%%) | gh5 groups=%d"%(
    thr,y.sum(),len(y),y.mean()*100,len(set(groups))))

SCORE_INTERNAL={"impact","gi_z","gi_p","pca_composite","impact_char","hand_score","rank","ranked",
                "gh7","gh6","gh5","lat","lon"}
SPATIAL={"lag_n","lag_sev_sum","lag_tier3_share","kde_sev"}
allfeat=[c for c in d.columns if c not in SCORE_INTERNAL and d[c].dtype!=object]
FULL=allfeat; TRANSFER=[c for c in allfeat if c not in SPATIAL]

def cv(feats,label):
    oof=np.zeros(len(d)); gkf=GroupKFold(n_splits=5)
    for tr,te in gkf.split(d,y,groups):
        m=lgb.LGBMClassifier(n_estimators=400,learning_rate=0.05,num_leaves=31,subsample=0.8,
            colsample_bytree=0.8,min_child_samples=20,reg_lambda=1.0,random_state=42,verbose=-1)
        m.fit(d.iloc[tr][feats],y[tr]); oof[te]=m.predict_proba(d.iloc[te][feats])[:,1]
    auc=roc_auc_score(y,oof); ap=average_precision_score(y,oof)
    print(f"  [{label:9s}] spatial-CV ROC-AUC={auc:.3f}  PR-AUC={ap:.3f}")
    return oof,auc,ap

oof_v,auc_v,_=cv(["n"],"volume")
oof_t,auc_t,ap_t=cv(TRANSFER,"transfer")
oof_f,auc_f,ap_f=cv(FULL,"full")

m=lgb.LGBMClassifier(n_estimators=400,learning_rate=0.05,num_leaves=31,subsample=0.8,
    colsample_bytree=0.8,min_child_samples=20,reg_lambda=1.0,random_state=42,verbose=-1)
m.fit(d[FULL],y)
imp=pd.Series(m.booster_.feature_importance(importance_type="gain"),index=FULL).sort_values(ascending=False)
imp.to_csv("/kaggle/working/fe_out/model_detect_importance.csv")
d.assign(p_hotspot=oof_f)[["gh7","impact","p_hotspot"]].to_csv("/kaggle/working/fe_out/model_detect_oof.csv",index=False)
m.booster_.save_model("/kaggle/working/fe_out/model_detect.txt")
print("\nTOP 10 DETECTION FEATURES (gain):"); print(imp.head(10).round(0).to_string())
print("\nSUMMARY ROC-AUC: volume=%.3f | transferable=%.3f | full=%.3f"%(auc_v,auc_t,auc_f))
